# GNN Decoder Experiments for 5G NR LDPC codes

This notebook reproduces the 5G NR LDPC code results in the paper [Graph Neural Networks for Channel Decoding](https://arxiv.org/pdf/2207.14742.pdf).

**Remark**: training can take several hours. This Sionna 2 / PyTorch port must be trained from scratch: the repository's TensorFlow checkpoint cannot be loaded by PyTorch.

This notebook requires Python 3.11+, PyTorch 2.9+, and [Sionna 2](https://nvlabs.github.io/sionna/). It intentionally does not import TensorFlow or Sionna RT.

Install the FEC-only dependency set in the selected Python 3.12 environment with `python -m pip install --upgrade torch sionna-no-rt ipykernel`.

In [ ]:
# Sionna 2.0 / PyTorch imports (Python 3.11+)
from pathlib import Path
import torch
import sionna.phy
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

# load required Sionna 2 components
from sionna.phy.fec.ldpc import LDPC5GEncoder, LDPC5GDecoder
from sionna.phy.utils.plotting import PlotBER

%load_ext autoreload
%autoreload 2
from gnn_torch import E2EModel, GNNBP, LDPC5GGNN, LinearEncoder, generate_pruned_pcm_5g, train_gnn, transfer_gnn_weights

torch.manual_seed(2)
device = torch.device(sionna.phy.config.device) # CPU on macOS; CUDA when available
print(f'Using device: {device}')

In [ ]:
# PyTorch selects Apple Metal (MPS) automatically when available.
print(f'PyTorch device: {device}')

## Define Hyperparameters and Load Code

We define all parameters as dictionary to support different architectures for different codes.


In [ ]:
#----- LDPC 5G -----
params={
    # --- Code Parameters ---
        "code": "5G-LDPC",
        "n": 140,
        "k": 60,
    # --- GNN Architecture ----
        "num_embed_dims": 16,
        "num_msg_dims": 16,
        "num_hidden_units": 48,
        "num_mlp_layers": 3,
        "num_iter": 10,
        "reduce_op": "sum",
        "activation": "relu",
        "clip_llr_to": 20,
        "use_attributes": False,
        "node_attribute_dims": 0,
        "msg_attribute_dims": 0,
        "return_infobits": False,
        "use_bias": True,        
    # --- Training ---- # 
        "batch_size": [128, 128, 128], # bs, iter, lr must have same dim
        "train_iter": [35000, 300000, 300000],
        "learning_rate": [5e-4, 1e-4, 1e-5],
        "ebno_db_train": [2, 8.],
        "ebno_db_eval": 2.,       
        "batch_size_eval": 1000, # batch size only used for evaluation during training
        "eval_train_steps": 1000, # evaluate model every N iters
    # --- Log ----
        "save_weights_iter": 10000, # save weights every X iters
        "run_name": "LDPC_5G_01", # name of the stored weights/logs
        "save_dir": "results/", # folder to store results
    # --- MC Simulation parameters ----
        "eval_num_iter": 10, # number of decoding iters to evaluate
        "mc_iters": 100,
        "mc_batch_size": 1000,
        "num_target_block_errors": 500,
        "ebno_db_min": 0.,
        "ebno_db_max": 4.5,
        "ebno_db_stepsize": 0.5,
        "eval_ns": [140, 280, 420, 280, 280, 280], # evaluate different lengths
        "eval_ks": [60, 120, 180, 120, 90, 150],    
        "sim_esno": False, # simulate results in EsN0
}

## Generate the 5G Decoding Graph

The 5G NR LDPC code is closely connected to rate-matching and, thus, a few graph pre-processing steps are required for our decoder.
We perform the following steps:
- Train including the first 2*Z punctured information bits (i.e., no puncturing for the training)
- Prune the parity-check matrix as much as possible and remove shortened positions from the graph

We would like to emphasize that the resulting trained GNN decoder is compliant with the standard code structure.

In [ ]:
# all codes must provide an encoder-layer and a pcm
if params["code"]=="5G-LDPC":    
    print("Loading 5G NR LDPC code") 

    k = params["k"]
    n = params["n"]
    
    encoder_5g = LDPC5GEncoder(k, n)
    decoder_5g = LDPC5GDecoder(encoder_5g,
                               num_iter=params["eval_num_iter"],
                               return_infobits=False,
                               prune_pcm=True)
    
    pcm,_ = generate_pruned_pcm_5g(decoder_5g, n)
    
    n_no_rm = pcm.shape[1]
    k_no_rm = pcm.shape[1] - pcm.shape[0]

    # create encoder without rate-matching
    u_ref = torch.eye(k, device=device)
    with torch.no_grad():
        c_ref = encoder_5g(u_ref)
    gm = torch.cat((u_ref[:, :2*encoder_5g.z], c_ref), dim=1)
    encoder_no_rm = LinearEncoder(gm)
    encoder_no_rm.to(device)

else:
    raise ValueError("Unknown code type")

## Simulate Baseline BER Performance

In [ ]:
ber_plot = PlotBER(f"GNN-based Decoding - {params['code']}, (k,n)=({k},{n})")
ebno_dbs = np.arange(params["ebno_db_min"],
                     params["ebno_db_max"]+1,
                     params["ebno_db_stepsize"])

In [ ]:
# uncoded QPSK
e2e_uncoded = E2EModel(None, None, k=100, n=100) # k and n are not relevant here
ber_plot.simulate(e2e_uncoded,
                  ebno_dbs=ebno_dbs,
                  batch_size=params["mc_batch_size"],
                  num_target_block_errors=params["num_target_block_errors"],
                  legend="Uncoded",
                  soft_estimates=True,
                  max_mc_iter=params["mc_iters"],
                  forward_keyboard_interrupt=False,
                  compile_mode=None);

### GNN-based Decoding without Rate-matching

For the training, we ignore the rate-matching and train the decoder
only for the given parity-check matrix.


In [ ]:
torch.manual_seed(2) # fix the seed to ensure stable convergence

# init the GNN decoder
gnn_decoder = GNNBP(pcm=pcm,
                     num_embed_dims=params["num_embed_dims"],
                     num_msg_dims=params["num_msg_dims"],
                     num_hidden_units=params["num_hidden_units"],
                     num_mlp_layers=params["num_mlp_layers"],
                     num_iter=params["num_iter"],
                     reduce_op=params["reduce_op"],
                     activation=params["activation"],
                     output_all_iter=True,
                     clip_llr_to=params["clip_llr_to"],
                     use_attributes=params["use_attributes"],
                     node_attribute_dims=params["node_attribute_dims"],
                     msg_attribute_dims=params["msg_attribute_dims"],
                     use_bias=params["use_bias"])
                     
e2e_gnn = E2EModel(encoder_no_rm, gnn_decoder, k_no_rm, n_no_rm, device=device)

In [ ]:
# Initialise model and print a PyTorch summary
e2e_gnn(1, 1.)
print(e2e_gnn)
print('Trainable parameters:', sum(p.numel() for p in e2e_gnn.parameters() if p.requires_grad))

In [ ]:
# and let's train the model...
train = True # training takes several hours; TensorFlow weights are not compatible
if train:    
    train_gnn(e2e_gnn, params)
else:
    checkpoint = Path(params['save_dir']) / f"{params['run_name']}_final.pt"
    if checkpoint.exists():
        e2e_gnn.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
        print(f'Loaded PyTorch weights from {checkpoint}')
    else:
        print('No PyTorch checkpoint found. Set train=True to train before evaluating the GNN.')

## Evaluate Final Performance

We now evaluate the performance for different codeword lengths and different rates.

In [ ]:
for idx, n_eval in enumerate(params["eval_ns"]):
    k_eval = params["eval_ks"][idx]
    
    # generate new code for each length
    encoder_5g_eval = LDPC5GEncoder(k_eval , n_eval)
    decoder_5g_eval = LDPC5GDecoder(encoder_5g_eval,
                                    num_iter=params["eval_num_iter"],
                                    return_infobits=params["return_infobits"])
    e2e_bp = E2EModel(encoder_5g_eval,
                      decoder_5g_eval,
                      k_eval,
                      n_eval,
                      return_infobits=params["return_infobits"],
                      es_no=params["sim_esno"] )

    ber_plot.simulate(e2e_bp,
                  ebno_dbs=ebno_dbs,
                  batch_size=params["mc_batch_size"],
                  num_target_block_errors=params["num_target_block_errors"],
                  legend=f"BP-{params['eval_num_iter']} n={n_eval}",
                  soft_estimates=True,
                  max_mc_iter=params["mc_iters"],
                  forward_keyboard_interrupt=False,
                  compile_mode=None);
    # instantiate new decoder for each number of iter (otherwise no retracing)
    gnn_decoder_rm_eval = LDPC5GGNN(encoder_5g_eval,
                     num_embed_dims=params["num_embed_dims"],
                     num_msg_dims=params["num_msg_dims"],
                     num_hidden_units=params["num_hidden_units"],
                     num_mlp_layers=params["num_mlp_layers"],
                     num_iter=params["eval_num_iter"],
                     reduce_op=params["reduce_op"],
                     activation=params["activation"],
                     output_all_iter=False,
                     clip_llr_to=params["clip_llr_to"],
                     use_attributes=params["use_attributes"],
                     node_attribute_dims=params["node_attribute_dims"],
                     msg_attribute_dims=params["msg_attribute_dims"],
                     return_infobits=params["return_infobits"],
                     use_bias=params["use_bias"])    
    # generate new model   
    model = E2EModel(encoder_5g_eval,
                     gnn_decoder_rm_eval,
                     k_eval,
                     n_eval,
                     return_infobits=params["return_infobits"],
                     es_no=params["sim_esno"],
                     device=device)
    model(1,1.) # init model
    # Copy PyTorch weights; legacy TensorFlow .npy weights cannot be reused.
    transfer_gnn_weights(gnn_decoder, model.decoder)

    # and run the BER simulations
    ber_plot.simulate(model,
                     ebno_dbs=ebno_dbs,
                     batch_size=params["mc_batch_size"],
                     num_target_block_errors=params["num_target_block_errors"],
                     legend=f"GNN-{model.decoder.num_iter} n={n_eval}",
                     soft_estimates=True,
                     max_mc_iter=params["mc_iters"],
                     forward_keyboard_interrupt=False,
                     compile_mode=None);

ber_plot(xlim=[0, 5], ylim=[1e-5, 0.2]) # show final figure

**Remark**: This figures shows the results of Fig. 5a and Fig. 5b in the paper (Fig. 5a is plotted in the Es/N0 domain)

In [ ]:
# save results for pgf plots
col_names = ["uncoded"]
for idx,n in enumerate(params["eval_ns"]):
    col_names.append("bp_k" + str(params["eval_ks"][idx]) + "_n" + str(n))
    col_names.append("gnn_k" + str(params["eval_ks"][idx]) + "_n" + str(n))
np.savetxt('results/ldpc_5g_ber.csv',
           np.column_stack([ebno_dbs] + ber_plot.ber),
           delimiter=',', header=','.join(['ebno_db'] + col_names), comments='')
print('Saved results/ldpc_5g_ber.csv')